# Infobae: exploración de la fuente (T2.2)

Lista las notas de Política con el sitemap diario (días 10 y 25 de cada mes desde el 10/12/2023), lee dos notas por día desde el JSON de Arc embebido en el HTML (`Fusion.globalContent`) y mide volumen, campos, notas en vivo y la relación entre la fecha de la URL y el día del sitemap.

No se usa la API del frontend (`/pf/api/`). Todas las requests pasan por `probe.py`: solo rutas permitidas por robots.txt, User-Agent identificable, 2 s entre requests y caché en `data/probe_cache/`. Con `PROBE_OFFLINE=1` el notebook corre solo desde la caché.

In [1]:
import re
from datetime import date, datetime, timedelta

import pandas as pd

import arc
import probe

p = probe.Probe()
LAST_MONTH = date(2026, 9, 1)  # último mes completo al explorar
pd.set_option("display.max_colwidth", 90)


def sitemap_url(day):
    return f"https://www.infobae.com/arc/outboundfeeds/sitemap/{day:%Y-%m-%d}/"


POLITICA = re.compile(r"^https://www\.infobae\.com/politica/(\d{4})/(\d{2})/(\d{2})/")


def is_politica(url):
    return bool(POLITICA.match(url))


def url_date(url):
    m = POLITICA.match(url)
    return date(int(m[1]), int(m[2]), int(m[3])) if m else None

## 1. Sitemaps diarios

In [2]:
counts = arc.count_sitemaps(p, arc.sample_days(LAST_MONTH), sitemap_url, is_politica)
print("Sitemaps sin respuesta 200:", int((counts["status"] != 200).sum()))
counts.drop(columns="politica_urls")

Sitemaps sin respuesta 200: 0


,day,status,total_urls,politica
0,2023-12-10,200,914,39
1,2023-12-25,200,797,6
2,2024-01-10,200,1636,30
3,2024-01-25,200,1960,28
4,2024-02-10,200,1154,17
...,...,...,...,...
63,2026-07-25,200,1004,10
64,2026-08-10,200,1277,16
65,2026-08-25,200,1727,24
66,2026-09-10,200,1870,28


## 2. Volumen estimado

Promedio de notas de Política de los dos días muestreados × días del mes. `spread_over_2x` marca meses donde un día tiene más del doble que el otro.

In [3]:
monthly = arc.monthly_estimate(counts)
display(monthly)
print("Total estimado desde el 10/12/2023 hasta el último mes completo:", monthly["estimated_notes"].sum())
print("Promedio diario de notas de Política:", round(counts["politica"].mean(), 1))

,month,sampled,estimated_notes,spread_over_2x
0,2023-12,"[39, 6]",495,True
1,2024-01,"[30, 28]",899,False
2,2024-02,"[17, 19]",522,False
3,2024-03,"[17, 23]",620,False
4,2024-04,"[39, 29]",1020,False
5,2024-05,"[26, 21]",728,False
6,2024-06,"[21, 24]",675,False
7,2024-07,"[18, 25]",666,False
8,2024-08,"[12, 12]",372,False
9,2024-09,"[33, 26]",885,False


Total estimado desde el 10/12/2023 hasta el último mes completo: 23079
Promedio diario de notas de Política: 22.6


## 3. Campos del esquema

In [4]:
notes = arc.read_notes(p, arc.pick_notes(counts))
print("Notas leídas:", len(notes), "| sin respuesta 200:", int((notes["status"] != 200).sum()))
arc.field_presence(notes)

Notas leídas: 136 | sin respuesta 200: 0


,column,source,present_share
0,id,hash of canonical_url (computed),1.000
1,medio,constant per outlet,NaN
2,url,"canonical_url (relative, prefixed with the site)",1.000
3,fecha_publicacion,"first_publish_date (UTC, converted to UTC-3)",1.000
4,titulo,headlines.basic,1.000
5,bajada,subheadlines.basic,1.000
6,cuerpo,"content_elements of type text, header and list (HTML)",1.000
7,autor,credits.by[].name,0.596
8,seccion,taxonomy.primary_section._id,1.000
9,tags,taxonomy.tags[].text,1.000


In [5]:
notes["cuerpo_words"].describe().round()

count     136.0
mean      926.0
std       427.0
min       472.0
25%       666.0
50%       811.0
75%      1018.0
max      2914.0
Name: cuerpo_words, dtype: float64

## 4. Notas en vivo

In [6]:
notes["live"] = notes["live_signals"].str.len() > 0
print("Proporción de notas en vivo:", round(notes["live"].mean(), 3))
display(pd.Series([s for signals in notes["live_signals"] for s in signals]).value_counts().rename("señal"))
display(notes.groupby("live")["cuerpo_words"].describe().round())
display(notes["subtype"].value_counts(dropna=False).rename("subtype"))

Proporción de notas en vivo: 0.022


url         3
headline    3
Name: señal, dtype: int64

,count,mean,std,min,25%,50%,75%,max
live,,,,,,,,
False,133.0,914.0,398.0,517.0,667.0,811.0,1011.0,2914.0
True,3.0,1439.0,1195.0,472.0,770.0,1069.0,1922.0,2775.0


subtype
comments=true     131
live-blogging       3
comments=false      2
Name: subtype, dtype: int64

## 5. Fecha de la URL vs día del sitemap

Diferencia en días entre la fecha que trae cada URL de Política y el día del sitemap que la lista (todas las URLs de los sitemaps muestreados).

In [7]:
arc.url_day_offsets(counts, url_date)

offset_days
-492       1
-39        1
-15        1
-7         1
-4         2
-2         6
-1        78
 0      1445
Name: urls, dtype: int64

## 6. `lastmod` del sitemap vs fechas de la nota

In [8]:
lastmod = {loc: mod for day in counts["day"] for loc, mod in arc.sitemap_entries(p.get(sitemap_url(day)).text)}
ok = notes[notes["status"] == 200].copy()
ok["lastmod"] = ok["url"].map(lastmod)
same = lambda a, b: (pd.to_datetime(ok[a], utc=True, format="ISO8601") - pd.to_datetime(ok[b], utc=True, format="ISO8601")).abs() < pd.Timedelta(seconds=1)
pd.Series({"lastmod == display_date": same("lastmod", "display_date").mean(),
           "lastmod == first_publish_date": same("lastmod", "fecha_publicacion").mean()}).round(3)

lastmod == display_date          0.265
lastmod == first_publish_date    0.243
dtype: float64

## Ejemplos

Cinco notas de la muestra, con el título recortado (no se publica el cuerpo).

In [9]:
display(notes.sample(5, random_state=1)[["sitemap_day", "fecha_publicacion", "url", "cuerpo_words"]]
        .assign(titulo=notes["titulo"].str[:80]))
print("Requests de red en esta corrida:", p.network_requests)

,sitemap_day,fecha_publicacion,url,cuerpo_words,titulo
42,2024-10-25,2024-10-25T21:30:47.2Z,https://www.infobae.com/politica/2024/10/25/el-jefe-de-hezbollah-en-america-latina-hab...,725,El jefe de Hezbollah en América Latina hablaba varios idiomas y se movía con ali
36,2024-09-10,2024-09-10T16:58:34.004Z,https://www.infobae.com/politica/2024/09/10/con-apoyo-de-la-cgt-los-gremios-aeronautic...,811,"Con apoyo de la CGT, los gremios aeronáuticos endurecen la pelea con el Gobierno"
93,2025-11-10,2025-11-10T14:17:01.665Z,https://www.infobae.com/politica/2025/11/10/daniel-arroyo-alerto-sobre-el-endeudamient...,701,Daniel Arroyo alertó sobre el endeudamiento familiar y la falta de redes comunit
5,2024-01-10,2024-01-10T12:25:54.256Z,https://www.infobae.com/politica/2024/01/10/un-jefe-comunal-de-santa-fe-le-envio-una-c...,772,Un jefe comunal de Santa Fe le envió una carta al presidente de Rusia con una in
65,2025-04-10,2025-04-10T13:21:05.724Z,https://www.infobae.com/politica/2025/04/10/el-segundo-de-kicillof-le-pedira-a-la-casa...,860,Kicillof le pidió al Gobierno el comando electoral: se llevó una muestra de “pre


Requests de red en esta corrida: 0
